# 07 - Batch and Realtime Inference

**Objectif :** charger le bundle final, réaliser une inférence batch sur le test untouched, puis simuler des prédictions temps réel sans API.

In [ ]:
import pandas as pd
import plotly.express as px

from credit_risk_lab.config.settings import settings

print(f"Project root: {settings.project_root}")
print(f"Environment: {settings.environment}")

## 1. Load external holdout

In [ ]:
from credit_risk_lab.infrastructure.data_sources import CsvLoanDataLoader

holdout_df = CsvLoanDataLoader(path=settings.raw_test_path).load()
holdout_df.head()

## 2. Load model bundle

In [ ]:
from credit_risk_lab.infrastructure.modeling import JoblibModelBundleRepository

repository = JoblibModelBundleRepository()
bundle = repository.load(settings.model_bundle_path)
bundle["metadata"]

## 3. Configure raw scorer and input validator

In [ ]:
from credit_risk_lab.application import InferenceInputValidator, RawLoanScorer

scorer = RawLoanScorer(bundle, threshold=settings.decision_threshold)
input_validator = InferenceInputValidator()

{
    "model_name": bundle["metadata"].get("model_name"),
    "serving_threshold": scorer.model_scorer.threshold,
    "test_rows_available": len(holdout_df),
}

## 4. Validate batch input sample

In [ ]:
batch_limit = 100
batch_input = holdout_df.head(batch_limit)
batch_validation = input_validator.validate(batch_input)

print(f"Rows received: {len(batch_input)}")
print(f"Rows accepted: {batch_validation.accepted_rows}")
print(f"Rows rejected: {batch_validation.rejected_count}")
print(f"Rows with warnings: {batch_validation.warning_count}")

print("Rejected rows")
display(batch_validation.rejected_rows.head(10))

print("Warning rows")
display(batch_validation.warning_rows.head(10))

## 5. Batch inference on valid test rows

In [ ]:
from credit_risk_lab.application import BatchInferenceRunner

submission_path = settings.reports_dir / "submission.csv"

batch_runner = BatchInferenceRunner(scorer, validator=input_validator)
batch_result = batch_runner.predict(
    batch_input,
    output_path=submission_path,
)

print(f"Batch rows scored: {len(batch_result.submission)}")
print(f"Batch rows rejected: {len(batch_result.rejected_rows)}")
print(f"Batch rows with warnings: {len(batch_result.warning_rows)}")
print(f"Submission saved to: {batch_result.output_path}")

display(batch_result.submission.head(10))

## 6. Batch inference risk distribution

In [ ]:
px.histogram(
    batch_result.submission,
    x="probability_of_risk",
    color="risk_band",
    nbins=30,
    title="Batch inference - predicted risk probabilities",
    template="plotly_white",
).show()

## 7. Realtime inference simulation without API

In [ ]:
from credit_risk_lab.application import RealtimeInferenceSimulator

realtime_limit = 5
simulator = RealtimeInferenceSimulator(
    scorer,
    validator=input_validator,
    min_pause_seconds=1,
    max_pause_seconds=5,
)

realtime_events = simulator.stream(
    holdout_df,
    limit=realtime_limit,
    request_prefix="loan-request",
    print_events=True,
)

display(realtime_events)

## 8. Realtime decision timeline

In [ ]:
fig = px.line(
    realtime_events,
    x="request_id",
    y="probability_of_risk",
    markers=True,
    title="Realtime simulation - request-level risk score",
    template="plotly_white",
)
fig.add_hline(
    y=settings.decision_threshold,
    line_dash="dash",
    line_color="black",
    annotation_text=f"threshold={settings.decision_threshold:.3f}",
)
fig.show()